In [1]:
# Seeds
import random
import numpy as np
import torch
import os

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print('Random seeds set for reproducibility')


Random seeds set for reproducibility


In [2]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import json
import glob
from tqdm import tqdm
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

print('Libraries imported successfully')

Libraries imported successfully


In [3]:
# Dataset class
class SimpsonsDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        try:
            image = Image.open(image_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (128, 128))
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

print('Dataset class defined')

Dataset class defined


In [4]:
#Unzip data
import zipfile

if os.path.exists('/content/archive/characters_train'):
    print('Data already exists at /content/archive/characters_train')
else:
    zip_path = '/content/Simpsons.zip'
    if os.path.exists(zip_path):
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                print(f'Found Simpsons.zip ({os.path.getsize(zip_path) / (1024*1024):.2f} MB)')
                zip_ref.testzip()
                print('Zip valid. Extracting...')
                zip_ref.extractall('/content/')
                print('Extraction complete!')
        except zipfile.BadZipFile:
            raise ValueError('Corrupted zip. Please re-upload.')
    else:
        raise FileNotFoundError('Please upload Simpsons.zip first')

Found Simpsons.zip (412.39 MB)
Zip valid. Extracting...
Extraction complete!


In [5]:
# Model architecture CNN with 4 convolutional blocks.
# Each block doubles the channel count (64→128→256→512) and includes batch normalization, ReLU activations, and max pooling.
#The classifier uses dropout(0.5/0.5/0.3) to prevent overfitting and has two fully connected layers (1024 and 512 units) before the final output layer with num_classes units.
class SimpsonsClassifier(nn.Module):
    def __init__(self, num_classes):
        super(SimpsonsClassifier, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(512 * 4 * 4, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

print('Model architecture defined')

Model architecture defined


In [6]:
# Load and prepare dataset
data_dir = '/content/archive/characters_train'

if not os.path.exists(data_dir):
    raise FileNotFoundError(f'Data directory not found: {data_dir}')

character_dirs = sorted([d for d in os.listdir(data_dir)
                         if os.path.isdir(os.path.join(data_dir, d))])
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(character_dirs)}
idx_to_class = {idx: cls_name for cls_name, idx in class_to_idx.items()}  #added reverse mapping
num_classes  = len(character_dirs)

print(f'Found {num_classes} character classes')
print(f'Classes: {character_dirs[:5]}...' if num_classes > 5 else f'Classes: {character_dirs}')

image_paths, labels = [], []
for character_name in character_dirs:
    character_path = os.path.join(data_dir, character_name)
    image_files = glob.glob(os.path.join(character_path, '*.jpg'))
    image_paths.extend(image_files)
    labels.extend([class_to_idx[character_name]] * len(image_files))

print(f'Total images: {len(image_paths)}')

# ensures balanced class split
train_paths, val_paths, train_labels, val_labels = train_test_split(
    image_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f'Train samples: {len(train_paths)}')
print(f'Validation samples: {len(val_paths)}')

class_counts = Counter(train_labels)
#save mapping so inference notebook can load class names without scanning data
with open('class_mapping.json', 'w') as f:
    json.dump({
        'class_to_idx': class_to_idx,
        'idx_to_class': {str(k): v for k, v in idx_to_class.items()}
    }, f, indent=2)
print('class_mapping.json saved')

Found 42 character classes
Classes: ['abraham_grampa_simpson', 'agnes_skinner', 'apu_nahasapeemapetilon', 'barney_gumble', 'bart_simpson']...
Total images: 16764
Train samples: 13411
Validation samples: 3353
class_mapping.json saved


In [7]:
# Training augmentation introduces geometric and colour variation so the model
# generalises to the unseen test set rather than memorising exact training crops
#tried focal loss  for minority class disbalance but
#both failed miserably dragging both f1 and accuracy down and skyrocketing loss

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

image_size = 224

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = SimpsonsDataset(train_paths, train_labels, transform=train_transform)
val_dataset   = SimpsonsDataset(val_paths,   val_labels,   transform=val_transform)

# WeightedRandomSampler for class imbalance
weights = [1.0 / class_counts[i] for i in range(num_classes)]
sample_weights = [weights[label] for label in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

batch_size   = 32
num_workers  = 2
train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,  num_workers=num_workers, pin_memory=True)

print('Data loaders created')

Data loaders created


In [8]:
# Model, loss, optimiser & scheduler
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model = SimpsonsClassifier(num_classes).to(device)
num_epochs = 50

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

# Adam with weight decay (L2 regularisation) and a conservative lr of 3e-4
optimizer = optim.Adam(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

# CosineAnnealingLR decays lr smoothly from 3e-4 down to eta_min=1e-6 over
# T_max epochs, avoiding a sudden drop and allowing fine convergence late in training.
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=1e-6
)

print('Model initialised')

Using device: cuda
Model initialised


In [9]:
#Training & validation functions
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    pbar = tqdm(train_loader, desc='Training')

    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        predicted = outputs.argmax(dim=1)

        running_loss += loss.item() * images.size(0)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        pbar.set_postfix({
            'loss': running_loss / total,
            'acc': 100 * correct / total
        })

    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    return running_loss / total, 100 * correct / total, macro_f1


def validate(model, val_loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        pbar = tqdm(val_loader, desc='Validation')

        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            predicted = outputs.argmax(dim=1)

            running_loss += loss.item() * images.size(0)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            pbar.set_postfix({
                'loss': running_loss / total,
                'acc': 100 * correct / total
            })

    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    return running_loss / total, 100 * correct / total, macro_f1

In [ ]:
# Saves the best model checkpoint based on validation Macro F1 rather than accuracy, so the saved weights are optimal for evaluation.
# Early stopping (patience=10) halts training if val F1 doesn't improve for 7 consecutive epochs, preventing overfitting and saving compute.
best_val_f1 = 0.0
patience = 7
no_improve = 0

print('\nStarting training...')

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    print('-' * 50)

    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_f1 = validate(model, val_loader, criterion, device)

    scheduler.step()

    print(f'Train  Loss: {train_loss:.4f} | Acc: {train_acc:.2f}% | Macro F1: {train_f1:.4f}')
    print(f'Val    Loss: {val_loss:.4f} | Acc: {val_acc:.2f}% | Macro F1: {val_f1:.4f}')

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        no_improve = 0

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_f1': val_f1,
            'num_classes': num_classes,
            'class_to_idx': class_to_idx,
            'idx_to_class': idx_to_class,
        }, 'model.pth')

        print(f'>>> Saved best model — Val Macro F1: {val_f1:.4f}')

    else:
        no_improve += 1

    if no_improve >= patience:
        print('Early stopping triggered')
        break

print(f'\nTraining complete! Best Val Macro F1: {best_val_f1:.4f}')
print('Model saved as model.pth')

#saved 2 best models on epoch 51 and 54 as after 55 epochs i assumed it would overfit as the gap
#between train and val f1 started widening and val f1 plateaued while train f1 kept improving, so i stopped training

# Download best model to local disk
from google.colab import files
import os
if os.path.exists('model.pth'):
    size_mb = os.path.getsize('model.pth') / 1e6
    print(f'Model file: model.pth | Size: {size_mb:.1f} MB | Best Macro F1: {best_val_f1:.4f} | Classes: {num_classes}')
    print('Starting download...')
    files.download('model.pth')
    print('Download triggered!')
else:
    print('model.pth not found — training may not have saved a checkpoint.')